### Tutorial  4

**2025-2026**

### Generative Adversarial Networks (GAN)

L'objectif de ce TP est de créer et manipuler un Generative Adversarial Network (GAN) afin de générer des images de chiffres manuscrits à partir d'une base de données d'exemples.

Vous aurez besoin des outils présents dans le fichier gan_tools_cuda.py que vous devrez copier dans votre répertoire Jupyter afin de pouvoir l'importer. Regardez bien les fonctions et classes définies dans ce fichier : il y a les codes permettant de télécharger la base de données d'images, de faire les conversions entre images et vecteurs, de générer les vecteurs de bruit à donner en entrée du générateur et de gérer l’affichage des résultats sous forme d'exemples d’images générées par le générateur et des indices de qualité du générateur et du discriminateur.

###### 1. Importation des librairies et de gan_tools_cuda.py

In [4]:
import torch
from torch import nn, optim
from torch.autograd.variable import Variable
import matplotlib.pyplot as plt

import gan_tools_cuda as gt

ModuleNotFoundError: No module named 'tensorboardX'

###### 2. Chargement des données d'apprentissage et création du "loader"

Le loader présentera au réseau les données par paquet de 100, dans un ordre aléatoire.

In [ ]:
# Load data
data = gt.mnist_data()
# Create loader with data, so that we can iterate over it
data_loader = torch.utils.data.DataLoader(data, batch_size=100, shuffle=True)
# Num batches
num_batches = len(data_loader)

###### 3. Création du discriminateur

Créez une classe Python DiscriminatorNet décrivant la structure du réseau discriminateur.

Le réseau est composé de trois couches cachées et d'une couche de sortie. Nous utiliserons le module "séquentiel" de Pytorch qui décrit l’ordre des différentes transformations à appliquer aux données dans chaque couche.

L'entrée du réseau est un vecteur avec 784 valeurs (une image 28x28), la sortie est une valeur unique, allant de 0 (fausse image) à 1 (vraie image) grâce à l'utilisation d'une fonction sigmoïde.

Les trois couches cachées utilisent une fonction LeakyReLU avec alpha = 0,2. Elles utilisent également une fonction Dropout pour mettre aléatoirement à zéro 30% des valeurs de sortie des neurones des couches cachées (cela améliore les performances du réseau en limitant le sur-apprentissage). Le nombre de neurones dans chaque couche cachée est respectivement de 1024, 512 et 256.

La classe possède une méthode "forward" permettant de calculer la sortie du réseau à partir d'un vecteur d'entrée (représentant une image).

**Question :** pourquoi utilise-t-on LeakyReLU plutôt que ReLU dans un GAN ? Calculez le nombre de paramètres du discriminateur.

In [ ]:
class DiscriminatorNet(torch.nn.Module):

    def __init__(self):
        super(DiscriminatorNet, self).__init__()
        n_features = 784
        n_out = 1

        self.hidden0 = nn.Sequential(
            nn.Linear(n_features, 1024),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3)
        )

        # A COMPLETER : couches hidden1 et hidden2

        self.out = nn.Sequential(
            torch.nn.Linear(256, n_out),
            torch.nn.Sigmoid()
        )

    def forward(self, x):
        x = self.hidden0(x)
        x = self.hidden1(x)
        x = self.hidden2(x)
        x = self.out(x)
        return x

###### 4. Création du générateur

Créez une classe Python GeneratorNet décrivant la structure du réseau générateur. Le réseau est composé de trois couches cachées et d'une couche de sortie. L'entrée du réseau est un vecteur avec 100 valeurs, la sortie est une image avec 784 valeurs (utilisez nn.Tanh() au lieu de nn.Sigmoid() pour convertir les valeurs de sortie). Les trois couches cachées utilisent une fonction LeakyReLU avec alpha = 0,2, mais pas de Dropout ici. Le nombre de neurones dans chaque couche cachée est respectivement de 256, 512 et 1024.

**Question :** pourquoi la sortie du générateur utilise-t-elle Tanh ? Quelle contrainte cela impose-t-il sur la normalisation des vraies images ?

In [ ]:
class GeneratorNet(torch.nn.Module):

    def __init__(self):
        super(GeneratorNet, self).__init__()

        # A COMPLETER

    def forward(self, x):

        # A COMPLETER
        pass

###### 5. Création du processus d'apprentissage

Initialisez les instances d'un discriminateur et d'un générateur. Créez deux optimiseurs de type Adam pour les deux réseaux. Définissez la fonction de perte : ici, nous choisissons une fonction d'entropie croisée binaire pour vérifier si le discriminateur trouve la bonne réponse.

In [ ]:
discriminator = DiscriminatorNet()
generator = GeneratorNet()
if torch.cuda.is_available():
    discriminator.cuda()
    generator.cuda()

# Optimizers
d_optimizer = optim.Adam(discriminator.parameters(), lr=0.0002)
g_optimizer = optim.Adam(generator.parameters(), lr=0.0002)

# Loss function
loss = nn.BCELoss()

Les fonctions train_generator et train_discriminator sont données ci-dessous. Expliquez les différentes étapes du processus d'apprentissage.

**Questions :**

a. Écrivez la fonction objectif du jeu min-max d'un GAN, $\min_G \max_D V(D, G)$, et montrez que la perte BCE utilisée dans `train_discriminator` correspond bien à la maximisation de $V$ par rapport à $D$.

b. Dans `train_generator`, la cible utilisée est `real_data_target` (des 1) alors que le générateur produit de fausses images. Expliquez ce choix : quelle quantité est minimisée ? Pourquoi est-ce préférable, en début d'apprentissage, à la minimisation directe de $\log(1 - D(G(z)))$ ? (Indication : étudiez le gradient lorsque $D(G(z)) \approx 0$.)

c. Pourquoi ne doit-on pas mettre à jour le générateur lors de l'entraînement du discriminateur ? Comment cela est-il garanti dans le code de la question 6 ?

In [ ]:
def train_discriminator(optimizer, real_data, fake_data):
    # Reset gradients
    optimizer.zero_grad()

    # 1.1 Train on Real Data
    prediction_real = discriminator(real_data)
    # Calculate error and backpropagate
    error_real = loss(prediction_real, gt.real_data_target(real_data.size(0)))
    error_real.backward()

    # 1.2 Train on Fake Data
    prediction_fake = discriminator(fake_data)
    # Calculate error and backpropagate
    error_fake = loss(prediction_fake, gt.fake_data_target(real_data.size(0)))
    error_fake.backward()

    # 1.3 Update weights with gradients
    optimizer.step()

    # Return error
    return error_real + error_fake, prediction_real, prediction_fake

In [ ]:
def train_generator(optimizer, fake_data):
    # 2. Train Generator
    # Reset gradients
    optimizer.zero_grad()
    # Sample noise and generate fake data
    prediction = discriminator(fake_data)
    # Calculate error and backpropagate
    error = loss(prediction, gt.real_data_target(prediction.size(0)))
    error.backward()
    # Update weights with gradients
    optimizer.step()
    # Return error
    return error

###### 6. Apprentissage et visualisation :

Pour chaque itération et pour chaque lot ("batch") de données, les étapes d'apprentissage sont les suivantes :

 - Transformez le lot de données en variables Torch à l'aide des fonctions Variable et gt.images_to_vectors. Ce sont les vraies données pour l'étape suivante.

 - Créez des vecteurs de bruit de taille 100 avec gt.noise (autant que de vraies images dans le lot), puis générez de fausses données avec le générateur.

 - Entraînez le discriminateur sur les fausses données et les données réelles, en utilisant l'optimiseur défini ci-dessus (d_optimizer).

 - Créez d'autres vecteurs de bruit, taille 100, avec gt.noise.

 - Entraînez le générateur sur les vecteurs de bruit.

 - **Enregistrez** à chaque lot l'erreur du discriminateur, l'erreur du générateur, ainsi que les moyennes de $D(x)$ et de $D(G(z))$ dans les listes `d_losses`, `g_losses`, `d_real`, `d_fake` (elles serviront à la question 7).

 - Visualisez un échantillon d'images générées avec gt.plot_gan. Cette fonction nécessite le numéro d'itération et de lot, ainsi que le nombre total de lots et le nom du générateur.

###### Attention :
Il se peut que le système se "bloque" dans une configuration où la sortie du générateur est la même quelle que soit l'entrée (*mode collapse*). Si toutes les images des exemples de sortie sont identiques après une dizaine d'itérations et que D(x) = 1 et D(G(z)) = 0, il faut relancer l'apprentissage (redémarrer le noyau).

In [ ]:
d_losses, g_losses, d_real, d_fake = [], [], [], []

for epoch in range(20):
    for n_batch, (real_batch,_) in enumerate(data_loader):

        # 1 Train Discriminator
        # 1.1 Prepare real data
        real_data = Variable(gt.images_to_vectors(real_batch))
        if torch.cuda.is_available(): real_data = real_data.cuda()

        # 1.2 Generate fake data with the Generator
        # A COMPLETER : créer des vecteurs bruit

        # A COMPLETER : générer des images avec le générateur

        # 1.3 Train Discriminator
        # A COMPLETER : entraîner le discriminateur

        # 2 Train Generator
        # 2.1 Generate noise
        # A COMPLETER : créer des vecteurs bruit à nouveau

        # 2.2 Train Generator
        # A COMPLETER : entraîner le générateur

        # A COMPLETER : enregistrer d_error, g_error, D(x) moyen et D(G(z)) moyen

        # Generate images from a fixed noise input and visualize the output
        gt.plot_gan(epoch, n_batch, num_batches, generator)

###### 7. Analyse des courbes d'apprentissage

Tracez sur une même figure les courbes `d_losses` et `g_losses` (lissez-les par une moyenne glissante sur 100 lots), puis sur une seconde figure les courbes `d_real` et `d_fake`.

Contrairement à un réseau classique, la perte du générateur ne décroît pas régulièrement : pourquoi ? Quelle valeur de $D(x)$ et de $D(G(z))$ obtiendrait-on théoriquement à l'équilibre du jeu ? Vos courbes s'en approchent-elles ?

###### 8. Exploration de l'espace latent

Tirez deux vecteurs de bruit $z_1$ et $z_2$ avec gt.noise. Générez les images correspondant à 10 points régulièrement espacés sur le segment $z_t = (1-t)\,z_1 + t\,z_2$, $t \in [0, 1]$, et affichez-les sur une même ligne (indication : `generator(z).view(-1, 28, 28)`, sans oublier `.detach().cpu()`).

Recommencez avec plusieurs paires $(z_1, z_2)$. Les transitions sont-elles progressives ? Qu'en déduisez-vous sur l'organisation de l'espace latent appris par le générateur ?

###### 9. Lissage des étiquettes (*one-sided label smoothing*)

Une technique classique pour stabiliser l'apprentissage consiste à remplacer, **pour le discriminateur uniquement**, la cible des vraies images 1 par 0,9.

Écrivez une fonction `train_discriminator_smooth` qui applique cette modification, réinitialisez les deux réseaux et leurs optimiseurs, puis relancez l'apprentissage sur 10 itérations. Comparez les images obtenues et les courbes de la question 7 avec celles de l'apprentissage initial. Pourquoi ne lisse-t-on pas aussi la cible des fausses images ?

###### 10. Exploration et optimisation

Modifiez la structure des deux réseaux (discriminateur et générateur) en ajoutant ou enlevant des couches cachées, en modifiant le nombre de neurones dans ces couches et en jouant sur les paramètres des fonctions LeakyReLU et Dropout.

Après avoir expliqué comment évaluer la qualité des images générées par le générateur (critères visuels **et** critères quantitatifs possibles), comparez la qualité des résultats obtenus avec les modifications que vous avez testées et essayez d’obtenir le meilleur résultat possible. Présentez vos essais dans un tableau récapitulatif.